# Master Exam Template: Classification Models
## Course Scope: Logistic Regression, Decision Tree Classifier, Random Forest Classifier
**Matches:** 7CS353 Lab Assignments 7, 8, 10  
**Marks Rubric (40 Marks):**
* Section 1: EDA [10 Marks]
* Section 2: Implementation (Logistic Regression Baseline) [10 Marks]
* Section 3: Evaluation (Confusion Matrix, Accuracy, Precision, Recall, F1) [10 Marks]
* Section 4: Modify Technique (Decision Tree & Random Forest with GridSearchCV) [10 Marks]

In [ ]:
# 1. Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

plt.rcParams['figure.figsize'] = (8, 4)
print("Libraries imported!")

In [ ]:
# Section 1: EDA (10 Marks) - Quick Exam Dataset Generation
np.random.seed(42)
n = 400

f1 = np.random.uniform(20, 80, n)
f2 = np.random.uniform(50, 150, n)
f3 = np.random.randint(1, 5, n)

z = 0.05 * f1 + 0.03 * f2 - 0.5 * f3 - 4.5
prob = 1 / (1 + np.exp(-z))
y = (prob > 0.5).astype(int)

df = pd.DataFrame({'Feature_1': f1, 'Feature_2': f2, 'Feature_3': f3, 'Target': y})

print("Dataset shape:", df.shape)
print("Class distribution:\n", df['Target'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x='Target', data=df, ax=axes[0], palette=['lightgreen', 'salmon'])
axes[0].set_title("Class Counts (0 vs 1)")
sns.heatmap(df.corr(), annot=True, cmap='Blues', ax=axes[1])
axes[1].set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# Section 2: Implementation (10 Marks) - Train/Test Split & Baseline
X = df.drop(columns=['Target'])
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
print("Baseline Logistic Regression trained!")

In [ ]:
# Section 3: Evaluation (10 Marks) - Confusion Matrix & Metrics
acc = accuracy_score(y_test, y_pred_lr)
prec = precision_score(y_test, y_pred_lr)
rec = recall_score(y_test, y_pred_lr)
f1 = f1_score(y_test, y_pred_lr)

res_lr = {'Model': 'Logistic Regression (Baseline)', 'Accuracy': round(acc, 4), 'Precision': round(prec, 4), 'Recall': round(rec, 4), 'F1-Score': round(f1, 4)}
display(pd.DataFrame([res_lr]))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(5, 3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred 0', 'Pred 1'], yticklabels=['Actual 0', 'Actual 1'])
plt.title("Confusion Matrix (Logistic Regression)")
plt.show()

In [ ]:
# Section 4: Modify Technique (10 Marks) - Decision Tree & Tuned Random Forest
dt = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train_scaled, y_train)
y_pred_dt = dt.predict(X_test_scaled)

res_dt = {'Model': 'Decision Tree (depth=3)', 'Accuracy': round(accuracy_score(y_test, y_pred_dt), 4), 'Precision': round(precision_score(y_test, y_pred_dt), 4), 'Recall': round(recall_score(y_test, y_pred_dt), 4), 'F1-Score': round(f1_score(y_test, y_pred_dt), 4)}

# Tuned Random Forest (GridSearchCV)
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), {'n_estimators': [50, 100], 'max_depth': [3, 5]}, cv=5, scoring='f1')
grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)

res_rf = {'Model': f'Tuned Random Forest {grid_rf.best_params_}', 'Accuracy': round(accuracy_score(y_test, y_pred_rf), 4), 'Precision': round(precision_score(y_test, y_pred_rf), 4), 'Recall': round(recall_score(y_test, y_pred_rf), 4), 'F1-Score': round(f1_score(y_test, y_pred_rf), 4)}

comparison_df = pd.DataFrame([res_lr, res_dt, res_rf])
print("=== Final Comparison Table ===")
display(comparison_df)